# 🎯 Day 05a: Transfer Learning — Pretrained Models & Fine-Tuning

---

## 🎯 What You'll Master Today
- Loading pretrained models (ResNet, MobileNet, etc.)
- Feature extraction vs fine-tuning
- Freezing/unfreezing layers
- Fine-tuning workflow & learning rate strategy

**Goal:** Transfer learning pipeline from memory — the most practical DL skill.

---

```
╔═══════════════════════════════════════════════════════════════════╗
║  Transfer Learning — Two Strategies                              ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  1. FEATURE EXTRACTION (freeze everything):                     ║
║     Pretrained model (frozen) → Your classifier head           ║
║     • Fast training, works with small data                      ║
║     • base.trainable = False                                    ║
║                                                                   ║
║  2. FINE-TUNING (unfreeze top layers):                          ║
║     Pretrained model (partially unfrozen) → Your head          ║
║     • Better accuracy, needs more data                          ║
║     • Low learning rate to not destroy pretrained weights       ║
║                                                                   ║
║  Workflow:                                                       ║
║     Phase 1: Feature extraction (train head only)               ║
║     Phase 2: Fine-tune (unfreeze top layers, low LR)           ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

# ============================================================
# 1. Loading Pretrained Models
# ============================================================

# Available in keras.applications:
# ResNet50, VGG16, MobileNetV2, EfficientNetB0, InceptionV3, etc.

base_model = keras.applications.MobileNetV2(
    weights='imagenet',     # pretrained on ImageNet
    include_top=False,      # remove classification head
    input_shape=(224, 224, 3),
)

print(f"Base model: {base_model.name}")
print(f"Layers: {len(base_model.layers)}")
print(f"Output shape: {base_model.output_shape}")
print(f"Trainable params: {sum(p.numpy().size for p in base_model.trainable_weights):,}")

print("\n💡 include_top=False removes the final Dense layers")
print("   This gives you feature maps to attach your own head")

In [ ]:
# ============================================================
# 2. Feature Extraction — Freeze Base, Train Head
# ============================================================

# Freeze the base model
base_model.trainable = False

# Build model with custom head
inputs = keras.Input(shape=(224, 224, 3))

# Preprocessing (MobileNetV2 expects [-1, 1] range)
x = keras.applications.mobilenet_v2.preprocess_input(inputs)

# Feature extraction (frozen)
x = base_model(x, training=False)  # training=False keeps BN in inference mode

# Classification head
x = layers.GlobalAveragePooling2D()(x)  # (batch, 7, 7, 1280) → (batch, 1280)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(5, activation='softmax')(x)  # 5 classes

model = keras.Model(inputs, outputs)

# Check trainable params
trainable = sum(p.numpy().size for p in model.trainable_weights)
non_trainable = sum(p.numpy().size for p in model.non_trainable_weights)
print(f"Trainable params:     {trainable:,}")     # only head
print(f"Non-trainable params: {non_trainable:,}")  # frozen base

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

print("\n💡 Phase 1: Only head trains. Fast, prevents destroying pretrained features.")
print("   training=False in base_model() → BN uses ImageNet statistics")

In [ ]:
# ============================================================
# 3. Fine-Tuning — Unfreeze Top Layers
# ============================================================

# After training the head, unfreeze some top layers of the base
base_model.trainable = True

# Freeze all layers except the last N
fine_tune_from = 100  # unfreeze layers from index 100 onwards
for layer in base_model.layers[:fine_tune_from]:
    layer.trainable = False

trainable_layers = sum(1 for l in base_model.layers if l.trainable)
frozen_layers = sum(1 for l in base_model.layers if not l.trainable)
print(f"Trainable layers: {trainable_layers}")
print(f"Frozen layers:    {frozen_layers}")

# Re-compile with LOWER learning rate (critical!)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),  # 100× lower!
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

# Then continue training:
# model.fit(train_ds, validation_data=val_ds, epochs=10)

print("\n💡 Phase 2: Unfreeze top layers, use VERY low LR (1e-5)")
print("   Too high LR → destroys pretrained features (catastrophic forgetting)")

In [ ]:
# ============================================================
# 4. Complete Transfer Learning Template
# ============================================================

def build_transfer_model(num_classes, input_shape=(224, 224, 3), 
                          backbone='mobilenetv2'):
    """Production transfer learning template."""
    
    # Choose backbone
    backbones = {
        'mobilenetv2': (keras.applications.MobileNetV2, 
                        keras.applications.mobilenet_v2.preprocess_input),
        'resnet50':    (keras.applications.ResNet50,
                        keras.applications.resnet50.preprocess_input),
        'efficientnetb0': (keras.applications.EfficientNetB0,
                           keras.applications.efficientnet.preprocess_input),
    }
    
    ModelClass, preprocess_fn = backbones[backbone]
    
    # Base model
    base = ModelClass(weights='imagenet', include_top=False, input_shape=input_shape)
    base.trainable = False  # start frozen
    
    # Build model
    inputs = keras.Input(shape=input_shape)
    x = preprocess_fn(inputs)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = keras.Model(inputs, outputs)
    return model, base

model, base = build_transfer_model(num_classes=10)
print(f"Model built: {sum(p.numpy().size for p in model.trainable_weights):,} trainable params")

print("""
Transfer Learning Workflow:
══════════════════════════
1. Load pretrained base (include_top=False)
2. Freeze base (base.trainable = False)
3. Add your head (GAP → Dense → Output)
4. Train head with normal LR (1e-3) for ~10 epochs
5. Unfreeze top layers of base
6. Re-compile with low LR (1e-5)
7. Fine-tune for ~10 more epochs
""")

In [ ]:
# ============================================================
# 5. Data Augmentation for Transfer Learning
# ============================================================

# Keras preprocessing layers (applied in the model graph)
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),      # ±10% rotation
    layers.RandomZoom(0.1),          # ±10% zoom
    layers.RandomContrast(0.1),      # ±10% contrast
])

# Add augmentation to the model
inputs = keras.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)  # only active during training
x = keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(5, activation='softmax')(x)

aug_model = keras.Model(inputs, outputs)
print("Model with augmentation built!")

print("""
╔══════════════════════════════════════════════════════╗
║  Transfer Learning Decision Guide                   ║
╠══════════════════════════════════════════════════════╣
║  Data size │ Similarity │ Strategy                  ║
║  ──────────│────────────│────────────────────────── ║
║  Small     │ Similar    │ Feature extraction only   ║
║  Small     │ Different  │ Feature extract from early║
║  Large     │ Similar    │ Fine-tune top layers      ║
║  Large     │ Different  │ Fine-tune many/all layers ║
╚══════════════════════════════════════════════════════╝

💡 Augmentation layers run INSIDE the model:
   - Active during training (model.fit)
   - Automatically disabled during inference (model.predict)
""")

---

## 🗺️ Transfer Learning Pattern Map

```
╔═══════════════════════════════════════════════════════════════════╗
║  TRANSFER LEARNING ESSENTIALS                                    ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  Load:  keras.applications.ModelName(weights='imagenet',        ║
║         include_top=False, input_shape=(...))                   ║
║                                                                   ║
║  Phase 1 — Feature Extraction:                                  ║
║    base.trainable = False, LR=1e-3                              ║
║                                                                   ║
║  Phase 2 — Fine-Tuning:                                         ║
║    Unfreeze top layers, LR=1e-5 (100× lower!)                  ║
║                                                                   ║
║  Head: GAP → BN → Dropout → Dense → Softmax                   ║
║  Augmentation: RandomFlip, RandomRotation, RandomZoom           ║
║  Always: training=False in base_model(x) for frozen base       ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | Feature extraction vs fine-tuning? | FE: freeze base, train head only. FT: unfreeze some layers, train with low LR. FE first, then FT |
| 2 | Why low LR for fine-tuning? | High LR destroys pretrained features (catastrophic forgetting). 1e-5 vs 1e-3 |
| 3 | What is include_top=False? | Removes the final classification layers. Returns feature maps. You add your own head |
| 4 | When to fine-tune vs feature extract? | Small+similar data → FE. Large data → FT. Different domain → FT more layers |
| 5 | Why training=False for frozen base? | BatchNorm layers use running statistics from pretraining. Without it, BN uses batch stats = bad |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Phase 1: freeze base, train head (LR=1e-3)          │
## │  • Phase 2: unfreeze top layers, fine-tune (LR=1e-5)   │
## │  • include_top=False → add your own head                │
## │  • training=False for frozen base (BN inference mode)   │
## │  • Augmentation layers inside model graph               │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## 🏁 ML/DL Code Fluency Notes COMPLETE!
## All 7 chapters, 28 notebooks finished.